In [ ]:
# Reduce CUDA memory fragmentation — set before any GPU operations
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Standard Protocol — Real-IAD

Runs all three models on Real-IAD under the standard protocol:
train on all 5 viewpoints, test on all 5 viewpoints.

Models: AnomalyDINO, Dinomaly, INP-Former
Dataset: Real-IAD (512px), all 30 categories
Protocol: Standard (all views)

In [ ]:
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
    !git -C {repo_path} submodule update --init
else:
    !git -C {repo_path} pull

sys.path.insert(0, repo_path)

!pip install anomalib==2.3.3 ADEval einops timm kornia -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 1: Load Real-IAD Data

In [ ]:
import importlib.util

# Load realiad_utils
spec = importlib.util.spec_from_file_location(
    "realiad_utils",
    f"{repo_path}/data/realiad_utils.py"
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
load_realiad_category = module.load_realiad_category

# Load realiad_dataset
spec2 = importlib.util.spec_from_file_location(
    "realiad_dataset",
    f"{repo_path}/data/realiad_dataset.py"
)
module2 = importlib.util.module_from_spec(spec2)
spec2.loader.exec_module(module2)
RealIADTorchDataset = module2.RealIADTorchDataset

# Load one category for smoke test
DATASET_ROOT = '/content/drive/MyDrive/datasets/realiad_512'
JSON_ROOT = f'{DATASET_ROOT}/realiad_jsons'

df = load_realiad_category(
    category_root=f'{DATASET_ROOT}/audiojack',
    json_path=f'{JSON_ROOT}/audiojack.json'
)

# Split into train (normal only) and test (all)
train_df = df[(df['split'] == 'train') & (df['label'] == 0)]
test_df = df[df['split'] == 'test']

print(f"Train (normal only): {len(train_df)}")
print(f"Test (all): {len(test_df)}")
print(f"Test label distribution:\n{test_df['label'].value_counts()}")

## Step 2: Run Dinomaly

In [ ]:
from anomalib.data import RealIAD
from anomalib.data.utils.split import TestSplitMode
from anomalib.models import Dinomaly
from anomalib.engine import Engine

DATASET_ROOT = '/content/drive/MyDrive/datasets'

datamodule = RealIAD(
    root=DATASET_ROOT,
    category='audiojack',
    resolution='512',
    json_path='realiad_512/realiad_jsons/{category}.json',
    train_batch_size=8,
    eval_batch_size=2,
    num_workers=2,
    test_split_mode=TestSplitMode.FROM_DIR,
)

model = Dinomaly()

engine = Engine(
    max_epochs=1,
    accelerator='gpu',
    devices=1,
)

print(f"Model: {model.__class__.__name__}")
print(f"Dataset: Real-IAD - audiojack")
print(f"Test split mode: FROM_DIR — includes normal test samples")
print("Ready to train")

In [ ]:
engine.fit(model=model, datamodule=datamodule)
print("Training complete")

In [ ]:
# Clear GPU memory between training and evaluation
# Prevents OOM errors on T4 (16GB) with large test sets
import torch
import gc
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

In [ ]:
engine.test(model=model, datamodule=datamodule)
print("Evaluation complete")